<table style="width: 100%; border-collapse: collapse; border: none; background: #fffbeb; border-left: 6px solid #f59e0b; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #78350f; font-size: 2em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        💡 El Libro de Cuentas de Doña Rosa: Qué es un Almacén de Datos 📒
      </h1>
      <p style="margin: 6px 0 0 0; color: #b45309; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Big Data
      </p>
      <p style="margin: 4px 0 0 0; color: #92400e; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #f59e0b; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        💡 Para Dummies • Módulo 02
      </span><br>
      <span style="color: #78350f; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #b45309; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

<div align="center" style="margin-top: 15px; margin-bottom: 15px;">
  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Big%20Data/02%20-%20Data%20Warehousing%20y%20Data%20Lake/Para%20Dummies/00_Data_Warehousing_Dummies.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>
  </a>
</div>

---
## ¿Qué vamos a aprender aquí? 🎈

Este cuaderno es la versión **«para no ingenieros»** del [cuaderno 00 del Módulo 02 (Data Warehousing, OLAP, ETL y esquema estrella)](../00_Data_Warehousing_OLAP_ETL_y_Esquema_Estrella.ipynb). Imagina a **doña Rosa**, dueña de una tienda en Duitama. Ella lleva **dos cosas distintas**:

| En la tienda de doña Rosa | En el mundo de los datos |
|---|---|
| **El cuaderno de la caja**: se anota cada venta **al momento**, rápido y a mano | La base de datos **de la operación** (OLTP): sirve para **registrar** |
| **El libro de balances de fin de mes**: ya ordenado, para **entender** cómo va el negocio | El **almacén de datos** (*data warehouse*): sirve para **analizar** |
| **Pasar del cuaderno al libro**: leer, corregir, descartar lo dudoso y copiar en limpio | El **ETL** (extraer, transformar, cargar) |
| **Fichas** de productos, de clientes y de fechas, y un renglón de venta que solo trae *códigos* | El **esquema estrella** (tabla de hechos + dimensiones) |
| **Resumir por mes y luego «abrir» un mes para ver el detalle** | Las operaciones **OLAP** (consolidar, detallar, rebanar, pivotar) |

Al terminar podrás explicar:
1. Por qué el cuaderno de la caja **no sirve** para responder «¿cómo nos fue en febrero?» sin ordenarlo antes.
2. Qué hace un **ETL**: **corregir** lo que se puede arreglar y **apartar con su motivo** lo que no.
3. Cómo se organiza un **almacén**: un renglón de venta con **códigos** y unas **fichas** que explican cada código.
4. Cómo se **resume y se detalla** un cubo con unas pocas líneas.

> 🔗 Si luego quieres ver el esquema con **DuckDB**, el ETL con **reglas de calidad**, la **historia de los clientes (SCD2)** y `ROLLUP`/`CUBE` en SQL: [cuaderno estándar](../00_Data_Warehousing_OLAP_ETL_y_Esquema_Estrella.ipynb). Antes de esto: el [Módulo 01](../../01%20-%20Procesamiento%20y%20Almacenamiento%20Distribuido/README.md).

---
## 1. Del Cuaderno de la Caja al Libro Limpio: el ETL 🧼

Estas son las **10 ventas** que doña Rosa anotó en su cuaderno (números **inventados**). Fíjate en los problemas de la vida real: una fecha escrita **al revés** (`06/01/2025` en vez de `2025-01-06`), una venta con **cantidad negativa**, un día que **no existe** (30 de febrero) y un renglón **repetido**.

El ETL tiene tres pasos, como una cocina: **E**xtraer (sacar del cuaderno), **T**ransformar (lavar, pelar, cortar) y **L**evar (**C**argar) al libro limpio. En la transformación hay dos tipos de decisión:

- **Corregir** lo que tiene arreglo **seguro**: la fecha `06/01/2025` es el 6 de enero; se reescribe bien.
- **Apartar a «cuarentena»** (una cajita de «revisar luego») lo que **no** se puede arreglar con seguridad, **anotando el motivo**: no se bota en silencio.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

cuaderno = pd.DataFrame({
    "id":       [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    "fecha":    ["2025-01-05", "2025-01-05", "06/01/2025", "2025-01-07", "2025-01-07", "2025-02-03", "2025-02-03", "2025-02-10", "2025-02-30", "2025-02-12"],
    "producto": ["leche", "papa", "leche", "queso", "arroz", "papa", "leche", "queso", "arroz", "papa"],
    "cantidad": [2, 5, 1, 1, 3, 4, -2, 2, 1, 6],
    "precio":   [2800, 2200, 2800, 18000, 4500, 2200, 2800, 18000, 4500, 2200],
    "cliente":  ["Ana", "Beto", "Ana", "Carla", "Beto", "Ana", "Carla", "Beto", "Ana", "Beto"],
})
cuaderno = pd.concat([cuaderno, cuaderno.iloc[[1]]], ignore_index=True)      # el renglon 2 aparece REPETIDO (11 renglones en total)


def etl(cuaderno):
    """Transformar: corregir lo seguro y apartar lo dudoso, con su motivo. Devuelve (limpio, cuarentena)."""
    c = cuaderno.copy()
    iso = pd.to_datetime(c.fecha, format="%Y-%m-%d", errors="coerce")
    latino = pd.to_datetime(c.fecha, format="%d/%m/%Y", errors="coerce")
    c["fecha_ok"] = iso.fillna(latino)                                          # CORREGIR: acepta los dos formatos de fecha
    c["motivo"] = ""
    c.loc[c.duplicated("id"), "motivo"] = "repetido"
    c.loc[(c.motivo == "") & c.fecha_ok.isna(), "motivo"] = "fecha imposible"
    c.loc[(c.motivo == "") & (c.cantidad <= 0), "motivo"] = "cantidad negativa"
    limpio = c[c.motivo == ""].assign(fecha=lambda d: d.fecha_ok.dt.strftime("%Y-%m-%d")).drop(columns=["fecha_ok", "motivo"]).reset_index(drop=True)
    return limpio, c[c.motivo != ""][["id", "fecha", "producto", "cantidad", "motivo"]]


libro_limpio, cuarentena = etl(cuaderno)
print(f"Recibo del ETL: entraron {len(cuaderno)} renglones | quedaron {len(libro_limpio)} limpios | {len(cuarentena)} a cuarentena")
print(cuarentena.to_string(index=False))
print("\nLa fecha corregida del renglon 3:", libro_limpio.loc[libro_limpio.id == 3, "fecha"].iloc[0])

assert len(cuaderno) == len(libro_limpio) + len(cuarentena)                      # nada se pierde sin explicacion: 11 = 8 + 3
assert sorted(cuarentena.motivo) == ["cantidad negativa", "fecha imposible", "repetido"]
assert libro_limpio.loc[libro_limpio.id == 3, "fecha"].iloc[0] == "2025-01-06"

---
**Lección:** entraron **11 renglones** y salieron **8 limpios** y **3 en cuarentena** (un repetido, un día imposible y una cantidad negativa). Lo importante es la **cuenta que cierra** (*11 = 8 + 3*): si no cerrara, el ETL se estaría «comiendo» ventas. Y ojo: **corregir no es lo mismo que acertar**; arreglar la fecha `06/01/2025` supone que doña Rosa siempre escribe *día/mes/año*. Esa **regla** se anota y alguien la aprueba.

---

## 2. El Libro de Balances: Hechos y Fichas (el Esquema Estrella) ⭐

En el libro de balances **cada venta** ocupa un renglón que **solo trae números y códigos**: *¿cuándo?* (código de fecha), *¿qué?* (código de producto), *¿quién?* (código de cliente) y *¿cuánto?* (el monto). Lo que **explica** cada código vive en una **ficha aparte**: la ficha del producto dice su categoría; la del cliente, su barrio; la de la fecha, su mes. Dibujado, parece una **estrella**: la tabla de **hechos** al centro y las **fichas** (dimensiones) alrededor. La ventaja: para saber «ventas por categoría» basta **unir** el renglón con la ficha del producto.

In [ ]:
# Las fichas (dimensiones): cada una explica un codigo
fichas_producto = pd.DataFrame({"cod_producto": [1, 2, 3, 4], "producto": ["leche", "papa", "queso", "arroz"], "categoria": ["lácteos", "tubérculos", "lácteos", "despensa"]})
fichas_cliente = pd.DataFrame({"cod_cliente": [1, 2, 3], "cliente": ["Ana", "Beto", "Carla"], "barrio": ["Centro", "Norte", "Centro"]})
fichas_fecha = pd.DataFrame({"fecha": sorted(libro_limpio.fecha.unique())}).assign(cod_fecha=lambda d: range(1, len(d) + 1), mes=lambda d: d.fecha.str[:7])

# El renglon de hechos: solo codigos y numeros
hechos = (libro_limpio.merge(fichas_producto, on="producto").merge(fichas_cliente, on="cliente").merge(fichas_fecha, on="fecha")
          .assign(monto=lambda d: d.cantidad * d.precio)[["id", "cod_fecha", "cod_producto", "cod_cliente", "cantidad", "monto"]].sort_values("id").reset_index(drop=True))
print("Tabla de HECHOS (solo codigos y numeros):"); print(hechos.to_string(index=False))

# Para responder una pregunta se UNE el renglon con la ficha
ventas_por_categoria = hechos.merge(fichas_producto, on="cod_producto").groupby("categoria").monto.sum()
print("\nVentas por categoria:", ventas_por_categoria.to_dict())
assert len(hechos) == len(libro_limpio) == 8 and ventas_por_categoria.sum() == hechos.monto.sum() == 108900

# Dibujo de la estrella
fig, ax = plt.subplots(figsize=(7.5, 3.8)); ax.set_xlim(0, 10); ax.set_ylim(0, 5); ax.axis("off"); ax.grid(False)
def caja(x, y, t, color, w=2.6):
    ax.add_patch(FancyBboxPatch((x, y), w, 0.9, boxstyle="round,pad=0.03,rounding_size=0.12", fc=color, ec="white", lw=2))
    ax.text(x + w / 2, y + 0.45, t, ha="center", va="center", color="white", fontweight="bold", fontsize=10)
caja(3.7, 2.05, "VENTAS (hechos)", "#1e3a8a")
caja(0.2, 3.7, "ficha de fechas", "#0ea5e9"); caja(7.2, 3.7, "ficha de productos", "#16a34a")
caja(0.2, 0.4, "ficha de clientes", "#f59e0b"); caja(7.2, 0.4, "(otra ficha: tiendas)", "#dc2626")
for p, q in [((2.0, 3.7), (4.2, 2.95)), ((8.3, 3.7), (6.0, 2.95)), ((2.0, 1.3), (4.2, 2.05)), ((8.3, 1.3), (6.0, 2.05))]:
    ax.annotate("", xy=q, xytext=p, arrowprops=dict(arrowstyle="-|>", lw=1.6, color="#475569"))
ax.set_title("El esquema estrella: un centro de ventas y fichas alrededor", fontweight="bold", fontsize=11)
plt.tight_layout(); plt.show()

---
**Lección:** el renglón de ventas guarda **códigos**, no textos repetidos («lácteos» no se escribe 500 veces: está **una vez** en la ficha). Si doña Rosa cambia el nombre de una categoría, lo corrige **en un solo lugar**. Y hay un detalle que importa: **si un cliente se muda de barrio**, ¿qué hacemos con las ventas viejas? Si **borramos** su barrio anterior, las ventas del pasado se «mudan» con él (¡se reescribe la historia!). Por eso los almacenes **guardan una ficha por cada versión** del cliente, con **fechas de vigencia** («vivió en el Centro hasta marzo; desde abril, en el Norte»). El cuaderno estándar lo implementa y lo verifica.

---

## 3. El Cubo: Resumir y Detallar 🧊

Con el libro limpio y las fichas, las preguntas del negocio son **resúmenes**. Las cuatro operaciones básicas, con el vocabulario del almacén:

| Lo que haces con las manos | Su nombre en el almacén | En pandas |
|---|---|---|
| «¿Cuánto vendí **por mes**?» (resumir) | ***Roll-up*** (consolidar) | `groupby("mes").sum()` |
| «Abre ese mes: ¿**por producto**?» (detallar) | ***Drill-down*** | `groupby(["mes", "producto"])` |
| «Solo lo que le vendí **a Ana**» | ***Slice*** (rebanada) | filtrar una condición |
| «Mes en filas, producto en columnas» | ***Pivot*** (rotar) | `pivot_table` |

In [ ]:
cubo = (hechos.merge(fichas_fecha, on="cod_fecha").merge(fichas_producto, on="cod_producto").merge(fichas_cliente, on="cod_cliente")
        [["id", "mes", "producto", "categoria", "cliente", "barrio", "cantidad", "monto"]])

por_mes = cubo.groupby("mes").monto.sum()                                         # ROLL-UP: resumir por mes
por_mes_producto = cubo.groupby(["mes", "producto"]).monto.sum()                  # DRILL-DOWN: detallar cada mes por producto
solo_ana = cubo[cubo.cliente == "Ana"]                                            # SLICE: fijar una condicion
tabla = cubo.pivot_table(index="mes", columns="producto", values="monto", aggfunc="sum", fill_value=0)   # PIVOT

print("Por mes:", por_mes.to_dict()); print("\nDetalle por mes y producto:"); print(por_mes_producto.to_string())
print("\nSolo Ana:", solo_ana.groupby("producto").monto.sum().to_dict()); print("\nPivot (mes x producto):"); print(tabla.to_string())

assert por_mes.sum() == cubo.monto.sum() == por_mes_producto.sum() == tabla.to_numpy().sum()      # resumir y detallar NO cambian el total
assert por_mes.to_dict() == {"2025-01": 50900, "2025-02": 58000}
assert solo_ana.monto.sum() == 2 * 2800 + 1 * 2800 + 4 * 2200

---
**Lección:** *resumir* (por mes) y *detallar* (por mes y producto) **no cambian el total**: solo lo **reparten** distinto; por eso un buen analista **verifica** que la suma del detalle coincide con el resumen. Rebanar (solo Ana) mira **una parte**; pivotar solo **gira la tabla**. Un almacén de verdad hace exactamente esto, pero con **millones** de renglones y consultas en SQL (`GROUP BY ROLLUP`).

---

## Resumen en una frase 🎯

> **Un almacén de datos es el «libro de balances» de una organización: se llena con un ETL que corrige lo seguro y aparta lo dudoso con su motivo, se organiza como una estrella (renglones de hechos con códigos y fichas que los explican) y se consulta resumiendo y detallando por cualquier ficha.**

### Lo que aprendiste hoy:
- ✅ **Cuaderno de caja frente a libro de balances:** una base **registra** (rápido, al momento) y el almacén **analiza** (ordenado, con historia).
- ✅ **ETL:** corregir lo que tiene arreglo seguro y apartar el resto **con su motivo**; el **recibo** (11 = 8 + 3) debe cerrar.
- ✅ **Estrella:** hechos al centro (códigos y números) y **fichas** alrededor; las fichas guardan **versiones** para no reescribir el pasado.
- ✅ **Cubo:** resumir (*roll-up*), detallar (*drill-down*), rebanar (*slice*) y pivotar; **el total no cambia**.

> 🎓 **Siguiente paso:** el [cuaderno estándar](../00_Data_Warehousing_OLAP_ETL_y_Esquema_Estrella.ipynb) construye todo con **DuckDB**, verifica cada regla de calidad contra los defectos sembrados y comprueba SQL contra pandas. Luego, [el lago de datos](01_Data_Lake_Dummies.ipynb).

---
##### 🛠️ Práctica 1: ¿Cuánto le vendió doña Rosa a Beto, mes por mes?

Con el `DataFrame` `cubo` (una fila por venta limpia, con las columnas `mes`, `cliente` y `monto`):

1. Filtra en `de_beto` solo las ventas de `"Beto"` (**rebanada**).
2. Calcula en `beto_por_mes` cuánto suman por mes (**resumir**).
3. Comprueba con `assert` que la suma de `beto_por_mes` es igual a la suma de `monto` de `de_beto`, y que el mes de **febrero** (`"2025-02"`) tiene **más** que enero para Beto.

In [ ]:
# Escribe tu código aquí

# de_beto = ...
# beto_por_mes = ...


<details>
<summary><b>💡 Haz clic aquí para ver la solución guiada...</b></summary>

```python
de_beto = cubo[cubo.cliente == "Beto"]
beto_por_mes = de_beto.groupby("mes").monto.sum()

assert beto_por_mes.sum() == de_beto.monto.sum()
assert beto_por_mes["2025-02"] > beto_por_mes["2025-01"]
print(beto_por_mes.to_dict())
```
</details>

---


---
### 🧠 ¿Ya lo tienes claro?

1. Con tus palabras: ¿por qué doña Rosa **no** usa el cuaderno de la caja para sacar el balance del mes sin ordenarlo antes? ¿Qué ventajas trae el «libro limpio»?
2. En el ETL, la fecha `06/01/2025` se **corrigió** y el día `2025-02-30` se **apartó**. ¿Qué diferencia hay entre los dos casos? ¿Qué podría salir mal si el ETL **corrigiera de más**?
3. Un cliente se muda de barrio. ¿Qué pasa con las ventas **viejas** si solo «sobrescribes» el barrio? ¿Qué se hace en un almacén para **no reescribir la historia**?

---
<div align="center">
  <p style="font-size: 0.9em; color: #78350f;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Big Data (Edición Didáctica Para Dummies)</i>
  </p>
</div>